In [ ]:
### 1. Base Model Training
    - Loss : Cross-Entropy Loss (VGG9, VGG5 - Cifar10)

In [ ]:
### Cross-Entropy Loss
ce_loss = nn.CrossEntropyLoss()
logits = model(inputs)
loss = ce_loss(logtis, labels)

probs = F.softmax(inputs, dim=-1)
_, preds = torch.max(probs, dim=-1)

## logits 의 크기와 probs 의 크기 관계가 같기 때문에 아래처럼 사용해도 무방
preds2 = torch.argmax(probs, dim=-1)
preds3 = torch.argmax(logits, dim=-1)

In [ ]:
### 1. Knowledge Distillation (Soft Targets)
    - Teacher's softmax output 학습 with Tempertuare & loss weight

## 확률 분포 차이의 Loss
    - Loss : KL Divergence Loss (Teacher, Student 의 softmax 결과 확률 분포 차이)
    - Loss = P(Ytrue) * ( log( P(Ytrue) ) - log( P(Ypred) ) )

## 지식 증류의 Soft Target Loss
    - Temperture 적용해서 scale 이 너무 커지지 않게 조정
    - Loss : Distillation Loss
    - Loss : D_kl(P || Q) / N * T^2                 # N : batch size, T : Temperture
             D_kl(P || Q) = T 로 나눈 KL Div Loss
             P, Q => P(x T) = softmax( x / T )

In [ ]:
## KL Div Loss
kl_loss = nn.KLDivLoss( reduction="batchmean" )

student_logits = stduent(inputs)
with torch.no_grad():
    teacher_logits = teacher(inputs)

student_log_probs = F.log_softmax( student_logits, dim=-1 )     # log softmax   => log(확률)
teacher_probs = F.softmax( teacher_logits, dim=-1 )             # only softmax  => 확률

loss = kl_loss( student_log_probs, teacher_probs )      # = mean of ( teacher_probs * ( teacher_probs.log() - student_log_probs ) )



## KL Div Loss 직접 계산
student_logits = stduent(inputs)
with torch.no_grad():
    teacher_logits = teacher(inputs)

student_probs = F.softmax( student_logits, dim=-1 )
teacher_probs = F.softmax( teacher_logits, dim=-1 )

loss = torch.sum( teacher_probs * ( teacher_probs.log() - student_probs.log() ) ).mean()



## KL Div Loss : 둘 다 log_softmax() 사용하는 법
kl_loss = nn.KLDivLoss( reduction="batchmean", log_target=True )    # log_target 을 True 하면 teacher 의 log_softmax() 값 사용 가능

student_logits = stduent(inputs)
with torch.no_grad():
    teacher_logits = teacher(inputs)

student_log_probs = F.log_softmax( student_logits, dim=-1 )
teacher_log_probs = F.log_softmax( teacher_logits, dim=-1 )

loss = kl_loss( student_log_probs, teacher_log_probs )

In [ ]:
## Distillation Loss
    - CE Loss : teacher, student 의 softmax 결과 값에 대한 loss (CE Loss)
    - KL Loss : Tempertuare 를 적용한 soft target 값에 대한 loss (KL Loss with T)
    - weighted sum of CE Loss & KL Loss

with torch.no_grad():
    teacher_logtis = teacher(inputs)
student_logits = student(inputs)

student_probs = F.softmax( student_logits / T, dim=-1 )
teacher_probs = F.softmax( teacher_logtis / T, dim=-1 )

soft_target_kl_loss = torch.sum( teacher_probs * ( teacher_probs.log() - student_probs.log() ) ) / student_probs.size(0) * (T**2)
ce_loss = nn.CrossEntropyLoss( student_logits, labels )

loss = soft_target_loss_wieght * soft_target_kl_loss + ce_loss_weight * ce_loss


## Distillation Loss 함수 사용 버전
with torch.no_grad():
    teacher_logtis = teacher(inputs)
student_logits = student(inputs)

student_log_probs = F.log_softmax( student_logits / T, dim=-1 )
teacher_probs = F.softmax( teacher_logtis / T, dim=-1 )

soft_target_kl_loss = nn.KLDivLoss( student_log_probs, teacher_probs, reduction="batchmean" ) * (T**2)
ce_loss = nn.CrossEntropyLoss( student_logits, labels )

loss = soft_target_loss_wieght * soft_target_kl_loss + ce_loss_weight * ce_loss

In [ ]:
### 2. Cosine Loss Minimization (Cosine Loss)
    - Conv Layers 의 최종 결과 생성되는 Hidden Feature 을 Embedding 한 vector 의 Cosine 유사도 측정
    - Cosine 유사도와 CE Loss 를 weighted sum 하여 최종 loss 계산

In [ ]:
## VGGCifar9_Cosine 의 forward 함수에서 hidden feature shape 변화
    - [B, 512, 2, 2]  => flatten(x, 1) => [B, 512 x 2 x 2] = [B, 2048] => avg_pool1d(kernel_size=2) => [B, 2048/2] = [B, 1024]

## VGGCifar5_Cosine 의 forward 함수에서 hidden feature shape 변화
    - [B, 256, 2, 2]  => flatten(x, 1) => [B, 256 x 2 x 2] = [B, 1024]

## Cosine 유사도 기반 지식 증류 Loss
with torch.no_grad():
    _, teacher_hidden_reperesentation = teacher( inputs )

student_logtis, student_hidden_representation = student( inputs )

cosine_loss = nn.CosineEmbeddingLoss( student_hidden_representation, teacher_hidden_reperesentation, torch.ones( inputs.size(0) ).to(device) )
ce_loss = nn.CrossEntropyLoss( student_logits, labels )

loss = cosine_loss_weight * cosine_loss + ce_loss_weight * ce_loss

In [ ]:
### 3. Intermediate Regressor (Regressor + MSE)
    - Conv Layers 의 최종 결과 생성되는 Hidden Feature map 의 오차를 누적한 Loss 사용
    - 작은 모델의 feature map 크기가 맞지 않으므로, 중간 regressor layer 로 크기를 맞춰준다
    - Regressor 의 MSE Loss 와 CE Loss 를 weighted sum 하여 최종 loss 계산

In [ ]:
### VGGCifar9_Regressor 의 forward 함수에서 리턴하는 conv_feature_map 의 shape
    - conv_feature_map : [B, 512, 2, 2]

## VGGCifar5_Regressor 에 추가한 regressor layer
    - nn.Sequential( nn.Conv2d( 256, 512, 3, padding=1, bias=False),    # 256 -> 512 채널로 늘려줌
                     nn.BatchNorm2d( 512 ))

## VGGCifar5_Regressor 의 forward 함수에서 리턴하는 conv_feature_map 의 shape
    - x : [B, 256, 2, 2] => self.regressor(x) => [B, 512, 2, 2]


## conv_feature_map 의 MSE Loss 기반 지식 증류 Loss
with torch.no_grad():
    _, teacher_feature_map = teacher( inputs )

student_logtis, student_feature_map = student( inputs )

feature_map_loss = nn.MSELoss( student_feature_map, teacher_feature_map )
ce_loss = nn.CrossEntropyLoss( student_logits, labels )

loss = feature_map_loss_weight * feature_map_loss + ce_loss_weight * ce_loss